# QC Summary

This notebook performs quality control checks on the final cardiovascular association table, covering genome build and coordinate handling, variant identifier availability and concordance, statistical field completeness, duplication, and GWAS Catalog trait classification across GWAS Catalog, HERMES, CARDIoGRAMplusC4D, and FinnGen.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
associations_path = Path("../data/final/association_results.csv")

out_qc_csv = Path("../data/final/qc_summary.csv")

In [3]:
associations_df = pd.read_csv(associations_path, low_memory=False, sep=";")

print("Association table shape:", associations_df.shape)

Association table shape: (238651, 35)


In [4]:
dataset_metadata = pd.DataFrame([
    {
        "source_dataset": "GWAS Catalog",
        "genome_build": "No coordinates reported",
        "coordinates_original_or_lifted": "Not available",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
    {
        "source_dataset": "HERMES",
        "genome_build": "GRCh37",
        "coordinates_original_or_lifted": "Original",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Main table excludes UK Biobank samples",
    },
    {
        "source_dataset": "CARDIoGRAMplusC4D",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Harmonized",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
    {
        "source_dataset": "FinnGen",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Original",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
])

dataset_metadata

,source_dataset,genome_build,coordinates_original_or_lifted,both_GRCh37_and_GRCh38_available,sample_overlap_flag
0,GWAS Catalog,No coordinates reported,Not available,No,Not applicable
1,HERMES,GRCh37,Original,No,Main table excludes UK Biobank samples
2,CARDIoGRAMplusC4D,GRCh38,Harmonized,No,Not applicable
3,FinnGen,GRCh38,Original,No,Not applicable


In [5]:
qc_df = associations_df.copy()

qc_df = qc_df.merge(
    dataset_metadata,
    on="source_dataset",
    how="left"
)

qc_df["p_value"] = pd.to_numeric(qc_df["p_value"], errors="coerce")
qc_df["effect_size"] = pd.to_numeric(qc_df["effect_size"], errors="coerce")
qc_df["standard_error"] = pd.to_numeric(qc_df["standard_error"], errors="coerce")
qc_df["odds_ratio"] = pd.to_numeric(qc_df["odds_ratio"], errors="coerce")
qc_df["allele_frequency"] = pd.to_numeric(qc_df["allele_frequency"], errors="coerce")

qc_df.shape

(238651, 39)

In [6]:
total_rows = len(qc_df)

def missing_result(column):
    if column not in qc_df.columns:
        return "column not available"
    missing = int(qc_df[column].isna().sum())
    present = int(qc_df[column].notna().sum())
    return f"{present}/{total_rows} rows available; {missing}/{total_rows} rows missing"

def metadata_summary(column):
    return "; ".join(
        dataset_metadata.apply(
            lambda row: f"{row['source_dataset']}: {row[column]}",
            axis=1
        )
    )

In [7]:
rows_by_source = (
    qc_df["source_dataset"]
    .value_counts(dropna=False)
    .reset_index()
)

rows_by_source.columns = ["source_dataset", "rows"]

rows_by_source

,source_dataset,rows
0,FinnGen,161563
1,CARDIoGRAMplusC4D,43409
2,HERMES,21252
3,GWAS Catalog,12427


In [8]:
genome_build_summary = (
    qc_df
    .groupby(["source_dataset", "genome_build"], dropna=False)
    .size()
    .reset_index(name="rows")
)

coordinate_handling_summary = (
    qc_df
    .groupby(["source_dataset", "coordinates_original_or_lifted"], dropna=False)
    .size()
    .reset_index(name="rows")
)

genome_build_summary, coordinate_handling_summary

(      source_dataset             genome_build    rows
 0  CARDIoGRAMplusC4D                   GRCh38   43409
 1            FinnGen                   GRCh38  161563
 2       GWAS Catalog  No coordinates reported   12427
 3             HERMES                   GRCh37   21252,
       source_dataset coordinates_original_or_lifted    rows
 0  CARDIoGRAMplusC4D                     Harmonized   43409
 1            FinnGen                       Original  161563
 2       GWAS Catalog                  Not available   12427
 3             HERMES                       Original   21252)

In [9]:
# Check repeated rsIDs within the same genome build for inconsistent positions.
position_check_df = qc_df.dropna(
    subset=["rsID", "region_assembly", "chromosome", "position"]
).copy()

position_check_df["variant_position_key"] = (
    position_check_df["chromosome"].astype(str)
    + ":"
    + position_check_df["position"].astype(str)
)

position_concordance = (
    position_check_df
    .groupby(["rsID", "region_assembly"])["variant_position_key"]
    .nunique()
    .reset_index(name="unique_positions_same_build")
)

position_conflicts = position_concordance[
    position_concordance["unique_positions_same_build"] > 1
]

# Check repeated rsIDs within the same genome build for inconsistent allele pairs.
# The allele pair is treated as unordered because effect allele direction can differ by dataset.
allele_check_df = qc_df.dropna(
    subset=["rsID", "region_assembly", "effect_allele", "other_allele"]
).copy()

allele_check_df["allele_pair"] = allele_check_df.apply(
    lambda row: "/".join(sorted([
        str(row["effect_allele"]).upper(),
        str(row["other_allele"]).upper()
    ])),
    axis=1
)

allele_concordance = (
    allele_check_df
    .groupby(["rsID", "region_assembly"])["allele_pair"]
    .nunique()
    .reset_index(name="unique_allele_pairs_same_build")
)

allele_conflicts = allele_concordance[
    allele_concordance["unique_allele_pairs_same_build"] > 1
]

print("Position conflicts:", len(position_conflicts))
print("Allele conflicts:", len(allele_conflicts))

Position conflicts: 0
Allele conflicts: 285


In [10]:
zero_pvalue_rows = qc_df[qc_df["p_value"] == 0].copy()

fully_duplicated_rows = int(qc_df[associations_df.columns].duplicated().sum())

repeated_rsid_summary = (
    qc_df
    .dropna(subset=["rsID"])
    .groupby("rsID")
    .agg(
        total_rows=("rsID", "size"),
        source_datasets=("source_dataset", lambda x: "; ".join(sorted(set(x.dropna())))),
        traits=("trait_name", lambda x: "; ".join(sorted(set(x.dropna())))),
    )
    .reset_index()
)

repeated_rsid_summary = repeated_rsid_summary[
    repeated_rsid_summary["total_rows"] > 1
].sort_values("total_rows", ascending=False)

print("Rows with p_value == 0:", len(zero_pvalue_rows))
print("Fully duplicated rows:", fully_duplicated_rows)
print("rsIDs appearing in more than one row:", len(repeated_rsid_summary))

Rows with p_value == 0: 455
Fully duplicated rows: 0
rsIDs appearing in more than one row: 50188


In [11]:
selected_trait_terms = [
    "heart failure",
    "coronary artery disease",
    "myocardial infarction",
]

gwas_df = qc_df[qc_df["source_dataset"] == "GWAS Catalog"].copy()

gwas_df["is_selected_trait"] = gwas_df["trait_name"].apply(
    lambda trait_name: (
        pd.notna(trait_name)
        and any(term in str(trait_name).lower() for term in selected_trait_terms)
    )
)

direct_cardiovascular_trait_rows = int(gwas_df["is_selected_trait"].sum())
broader_trait_rows = int((~gwas_df["is_selected_trait"]).sum())

print("Direct cardiovascular GWAS Catalog rows:", direct_cardiovascular_trait_rows)
print("Broader or non-selected GWAS Catalog rows:", broader_trait_rows)

Direct cardiovascular GWAS Catalog rows: 17
Broader or non-selected GWAS Catalog rows: 12410


In [12]:
qc_df["calculated_odds_ratio"] = np.exp(qc_df["effect_size"])

qc_df["or_absolute_difference"] = (
    qc_df["odds_ratio"] - qc_df["calculated_odds_ratio"]
).abs()

rows_with_effect_size = int(qc_df["effect_size"].notna().sum())
rows_with_odds_ratio = int(qc_df["odds_ratio"].notna().sum())
rows_with_effect_size_but_missing_odds_ratio = int(
    (qc_df["effect_size"].notna() & qc_df["odds_ratio"].isna()).sum()
)

max_or_difference = qc_df["or_absolute_difference"].max(skipna=True)

print("Rows with effect_size:", rows_with_effect_size)
print("Rows with odds_ratio:", rows_with_odds_ratio)
print("Rows with effect_size but missing odds_ratio:", rows_with_effect_size_but_missing_odds_ratio)
print("Maximum OR difference from exp(effect_size):", max_or_difference)

Rows with effect_size: 226223
Rows with odds_ratio: 226224
Rows with effect_size but missing odds_ratio: 0
Maximum OR difference from exp(effect_size): 5.03997237721876e-07


In [13]:
qc_rows = [
    {
        "check": "Source dataset recorded for each row",
        "result": (
            f"All {total_rows} association rows have source_dataset recorded"
            if qc_df["source_dataset"].notna().all()
            else f"{int(qc_df['source_dataset'].isna().sum())}/{total_rows} rows missing source_dataset"
        ),
    },
    {
        "check": "Genome build used by each dataset recorded",
        "result": metadata_summary("genome_build"),
    },
    {
        "check": "Coordinates recorded as original or lifted over",
        "result": metadata_summary("coordinates_original_or_lifted"),
    },
    {
        "check": "Availability of both GRCh37 and GRCh38 coordinates recorded",
        "result": metadata_summary("both_GRCh37_and_GRCh38_available"),
    },
    {
        "check": "rsIDs, alleles, and positions checked for availability and concordance",
        "result": (
            f"{int(qc_df['rsID'].isna().sum())} rows missing rsID; "
            f"{int(qc_df['chromosome'].isna().sum())} rows missing chromosome; "
            f"{int(qc_df['position'].isna().sum())} rows missing position; "
            f"{int(qc_df['effect_allele'].isna().sum())} rows missing effect_allele; "
            f"{int(qc_df['other_allele'].isna().sum())} rows missing other_allele; "
            f"{len(position_conflicts)} rsID-position conflicts within same build; "
            f"{len(allele_conflicts)} rsID-allele-pair conflicts within same build"
        ),
    },
    {
        "check": "Effect allele and non-effect allele clearly defined",
        "result": (
            f"effect_allele: {int(qc_df['effect_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['effect_allele'].isna().sum())} missing; "
            f"other_allele: {int(qc_df['other_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['other_allele'].isna().sum())} missing"
        ),
    },
    {
        "check": "p-values, effect sizes, standard errors, and allele frequencies available",
        "result": (
            f"p_value: {missing_result('p_value')}; "
            f"effect_size: {missing_result('effect_size')}; "
            f"standard_error: {missing_result('standard_error')}; "
            f"allele_frequency: {missing_result('allele_frequency')}"
        ),
    },
    {
        "check": "p-values reported as zero checked",
        "result": f"{len(zero_pvalue_rows)} rows with p_value = 0",
    },
    {
        "check": "Same variant appearing multiple times across traits or datasets checked",
        "result": f"{len(repeated_rsid_summary)} rsIDs appear in more than one association row",
    },
    {
        "check": "Fully duplicated rows checked",
        "result": f"{fully_duplicated_rows} fully duplicated rows",
    },
    {
        "check": "GWAS Catalog associations checked for direct cardiovascular traits or broader molecular/biomarker traits",
        "result": (
            f"{direct_cardiovascular_trait_rows} direct cardiovascular GWAS Catalog rows; "
            f"{broader_trait_rows} broader or non-selected GWAS Catalog rows"
        ),
    },
    {
        "check": "Potential sample overlap clearly flagged",
        "result": metadata_summary("sample_overlap_flag"),
    },
    {
        "check": "Odds ratios available or derivable from effect size",
        "result": (
            f"{rows_with_effect_size} rows with effect_size; "
            f"{rows_with_odds_ratio} rows with odds_ratio; "
            f"{rows_with_effect_size_but_missing_odds_ratio} rows with effect_size but missing odds_ratio; "
            f"maximum OR difference from exp(effect_size): {max_or_difference:.6f}"
            if max_or_difference is not None and not np.isnan(max_or_difference)
            else f"{rows_with_effect_size} rows with effect_size; "
                 f"{rows_with_odds_ratio} rows with odds_ratio; "
                 f"{rows_with_effect_size_but_missing_odds_ratio} rows with effect_size but missing odds_ratio; "
                 f"maximum OR difference from exp(effect_size): not calculable"
        ),
    },
]

qc_summary_df = pd.DataFrame(qc_rows)

qc_summary_df

,check,result
0,Source dataset recorded for each row,All 238651 association rows have source_datase...
1,Genome build used by each dataset recorded,GWAS Catalog: No coordinates reported; HERMES:...
2,Coordinates recorded as original or lifted over,GWAS Catalog: Not available; HERMES: Original;...
3,Availability of both GRCh37 and GRCh38 coordin...,GWAS Catalog: No; HERMES: No; CARDIoGRAMplusC4...
4,"rsIDs, alleles, and positions checked for avai...",5234 rows missing rsID; 12427 rows missing chr...
5,Effect allele and non-effect allele clearly de...,"effect_allele: 238651/238651 available, 0 miss..."
6,"p-values, effect sizes, standard errors, and a...",p_value: 238651/238651 rows available; 0/23865...
7,p-values reported as zero checked,455 rows with p_value = 0
8,Same variant appearing multiple times across t...,50188 rsIDs appear in more than one associatio...
9,Fully duplicated rows checked,0 fully duplicated rows


In [14]:
qc_summary_df.to_csv(out_qc_csv, index=False, sep=";")